# Essential DataFrame Methods & Analysis
## Data Analysis, Cleaning, and Transformation Techniques

This session dives into the **most practical methods** for analyzing and transforming DataFrames. These techniques are used daily in real data science work!

---

## 📚 Learning Objectives

By the end of this session, you will be able to:

1. Sort data with `.sort_values()` and `.rank()`
2. Analyze unique values with `.value_counts()` and `.unique()`
3. Find duplicates and remove them with `.drop_duplicates()`
4. Fill missing data comprehensively with `.fillna()` and interpolation
5. Apply functions to data with `.apply()` and  `.applymap()`
6. Calculate statistical relationships with `.corr()` and `.cov()`
7. Find extreme values with `.nlargest()` and `.nsmallest()`
8. Summarize and aggregate data effectively

---

## Getting Started

In [57]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [58]:
# Create sample datasets locally (no external URLs needed)
import random
random.seed(42)
np.random.seed(42)

# Netflix Dataset
netflix = pd.DataFrame({
    'title': ['Breaking Bad', 'The Office', 'Stranger Things', 'Friends', 'The Crown', 'Narcos', 
              'The Good Place', 'Chernobyl', 'Mindhunter', 'Dark', 'The Witcher', 'Ozark',
              'Peaky Blinders', 'Handmaid\'s Tale', 'Bodyguard'] * 4,
    'type': [random.choice(['Movie', 'TV Show']) for _ in range(60)],
    'release_year': [random.randint(2015, 2024) for _ in range(60)],
    'rating': [random.choice(['TV-MA', 'TV-14', 'PG-13', 'R', 'PG']) for _ in range(60)]
})

# Superstore Dataset
superstore = pd.DataFrame({
    'Order ID': range(1001, 1101),
    'Product Name': [f'Prod_{random.choice(["A","B","C","D"])}' for _ in range(100)],
    'Region': [random.choice(['North', 'South', 'East', 'West']) for _ in range(100)],
    'Sales': np.random.uniform(10, 500, 100),
    'Profit': np.random.uniform(-50, 150, 100),
    'Quantity': np.random.randint(1, 20, 100),
    'Category': [random.choice(['Electronics', 'Furniture', 'Office Supplies']) for _ in range(100)]
})

# Google Play Store Dataset
googleplay = pd.DataFrame({
    'App': [f'App_{i}' for i in range(80)],
    'Category': [random.choice(['Games', 'Productivity', 'Social', 'News', 'Music']) for _ in range(80)],
    'Rating': np.random.uniform(2, 5, 80),
    'Reviews': np.random.randint(100, 100000, 80),
    'Installs': np.random.randint(1000, 1000000, 80)
})

print("Datasets loaded successfully!")
print(f"Netflix: {netflix.shape}")
print(f"Superstore: {superstore.shape}")
print(f"Google Play Store: {googleplay.shape}")

Datasets loaded successfully!
Netflix: (60, 4)
Superstore: (100, 7)
Google Play Store: (80, 5)


---

## Part 1: Sorting Data

### 1A: Basic Sorting with sort_values()

In [59]:
# Sort by a single column (ascending)
print("Netflix titles sorted alphabetically:")
sorted_asc = netflix.sort_values('title').head()
print(sorted_asc[['title', 'type', 'release_year']])

Netflix titles sorted alphabetically:
           title   type  release_year
29     Bodyguard  Movie          2023
44     Bodyguard  Movie          2017
14     Bodyguard  Movie          2020
59     Bodyguard  Movie          2019
0   Breaking Bad  Movie          2024


In [60]:
# Sort descending
print("Most recent Netflix content:")
sorted_desc = netflix.sort_values('release_year', ascending=False).head()
print(sorted_desc[['title', 'type', 'release_year']])

Most recent Netflix content:
              title     type  release_year
0      Breaking Bad    Movie          2024
19        The Crown  TV Show          2024
32  Stranger Things  TV Show          2024
34        The Crown  TV Show          2024
47  Stranger Things    Movie          2024


In [61]:
# Sort by multiple columns
print("Sort by type, then by year (descending):")
multi_sort = netflix.sort_values(['type', 'release_year'], ascending=[True, False]).head(10)
print(multi_sort[['title', 'type', 'release_year']])

Sort by type, then by year (descending):
              title   type  release_year
0      Breaking Bad  Movie          2024
47  Stranger Things  Movie          2024
29        Bodyguard  Movie          2023
39             Dark  Movie          2023
5            Narcos  Movie          2022
6    The Good Place  Movie          2021
15     Breaking Bad  Movie          2021
35           Narcos  Movie          2021
49        The Crown  Movie          2021
10      The Witcher  Movie          2020


### 1B: Using rank() for Ranking

In [62]:
# Rank values
sales_data = pd.DataFrame({
    'Product': ['A', 'B', 'C', 'D', 'E'],
    'Sales': [100, 250, 100, 450, 300]
})

print("Original data:")
print(sales_data)

# Add rank column
sales_data['Rank'] = sales_data['Sales'].rank()
print("\nWith rank (handles ties):")
print(sales_data)

# Method for ranking (average, min, max, first, dense)
sales_data['Rank_Min'] = sales_data['Sales'].rank(method='min')
print("\nWith rank(method='min'):")
print(sales_data[['Product', 'Sales', 'Rank', 'Rank_Min']])

Original data:
  Product  Sales
0       A    100
1       B    250
2       C    100
3       D    450
4       E    300

With rank (handles ties):
  Product  Sales  Rank
0       A    100   1.5
1       B    250   3.0
2       C    100   1.5
3       D    450   5.0
4       E    300   4.0

With rank(method='min'):
  Product  Sales  Rank  Rank_Min
0       A    100   1.5       1.0
1       B    250   3.0       3.0
2       C    100   1.5       1.0
3       D    450   5.0       5.0
4       E    300   4.0       4.0


> 💡 **Best Practice**: Use `.rank()` when you need to preserve original data order or when ties matter. For simple sorting, `.sort_values()` is cleaner.

---

## Part 2: Value Counts & Unique Values

### 2A: value_counts() - Frequency Analysis

In [63]:
# Most common Netflix types
print("Netflix content type distribution:")
type_counts = netflix['type'].value_counts()
print(type_counts)

# With percentages
print("\nPercentage distribution:")
pct = netflix['type'].value_counts(normalize=True) * 100
print(pct.round(2))

Netflix content type distribution:
type
Movie      35
TV Show    25
Name: count, dtype: int64

Percentage distribution:
type
Movie      58.33
TV Show    41.67
Name: proportion, dtype: float64


In [64]:
# Top 10 most common ratings
print("Top 10 ratings:")
ratings = netflix['rating'].value_counts().head(10)
print(ratings)

Top 10 ratings:
rating
PG       14
TV-14    14
TV-MA    12
PG-13    11
R         9
Name: count, dtype: int64


In [65]:
# Include NaN in counts
print("Ratings (including NaN):")
ratings_with_nan = netflix['rating'].value_counts(dropna=False).head()
print(ratings_with_nan)

Ratings (including NaN):
rating
PG       14
TV-14    14
TV-MA    12
PG-13    11
R         9
Name: count, dtype: int64


### 2B: unique() & nunique()

In [66]:
# Get unique values
unique_types = netflix['type'].unique()
print(f"Unique types: {unique_types}")
print(f"Number of unique types: {netflix['type'].nunique()}")

# For numeric data
unique_years = netflix['release_year'].dropna().unique()
print(f"\nUnique years count: {netflix['release_year'].nunique()}")
print(f"Year range: {netflix['release_year'].min():.0f} - {netflix['release_year'].max():.0f}")

Unique types: ['Movie' 'TV Show']
Number of unique types: 2

Unique years count: 10
Year range: 2015 - 2024


---

## Part 3: Handling Duplicates

### 3A: Finding Duplicates

In [67]:
# Create sample data with duplicates
sample_data = pd.DataFrame({
    'Name': ['Alice', 'Bob', 'Alice', 'Charlie', 'Bob'],
    'Age': [25, 30, 25, 35, 30],
    'City': ['NYC', 'LA', 'NYC', 'Chicago', 'LA']
})

print("Sample data:")
print(sample_data)

# Find duplicates
print("\nDuplicate rows:")
print(sample_data[sample_data.duplicated()])

# Include first occurrence
print("\nAll duplicates (including first):")
print(sample_data[sample_data.duplicated(keep=False)].sort_values('Name'))

Sample data:
      Name  Age     City
0    Alice   25      NYC
1      Bob   30       LA
2    Alice   25      NYC
3  Charlie   35  Chicago
4      Bob   30       LA

Duplicate rows:
    Name  Age City
2  Alice   25  NYC
4    Bob   30   LA

All duplicates (including first):
    Name  Age City
0  Alice   25  NYC
2  Alice   25  NYC
1    Bob   30   LA
4    Bob   30   LA


In [68]:
# Check for duplicates in specific columns
print("Rows with duplicate names:")
dup_names = sample_data[sample_data.duplicated(subset=['Name'], keep=False)]
print(dup_names)

Rows with duplicate names:
    Name  Age City
0  Alice   25  NYC
1    Bob   30   LA
2  Alice   25  NYC
4    Bob   30   LA


### 3B: Removing Duplicates

In [69]:
# Remove all duplicates (keep first)
cleaned = sample_data.drop_duplicates()
print(f"Original: {len(sample_data)} rows")
print(f"Cleaned: {len(cleaned)} rows")
print("\nCleaned data:")
print(cleaned)

Original: 5 rows
Cleaned: 3 rows

Cleaned data:
      Name  Age     City
0    Alice   25      NYC
1      Bob   30       LA
3  Charlie   35  Chicago


In [70]:
# Remove duplicates based on specific columns
cleaned_by_name = sample_data.drop_duplicates(subset=['Name'])
print("Remove duplicate names (keep first):")
print(cleaned_by_name)

# Keep last occurrence instead
cleaned_last = sample_data.drop_duplicates(subset=['Name'], keep='last')
print("\nRemove duplicate names (keep last):")
print(cleaned_last)

Remove duplicate names (keep first):
      Name  Age     City
0    Alice   25      NYC
1      Bob   30       LA
3  Charlie   35  Chicago

Remove duplicate names (keep last):
      Name  Age     City
2    Alice   25      NYC
3  Charlie   35  Chicago
4      Bob   30       LA


> 🔍 **Real-World Use Case**: In E-commerce data, duplicates often indicate customer transactions recorded twice. Removing them is crucial before analysis!

---

## Part 4: Handling Missing Data (Comprehensive)

### 4A: Identifying Missing Data

In [71]:
# Summary of missing data
missing = pd.DataFrame({
    'Missing Count': netflix.isnull().sum(),
    'Missing %': (netflix.isnull().sum() / len(netflix) * 100).round(2)
})
missing = missing[missing['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print("Missing data summary:")
print(missing)

Missing data summary:
Empty DataFrame
Columns: [Missing Count, Missing %]
Index: []


### 4B: Filling Missing Data

In [72]:
# Create example data with missing values
test_data = pd.DataFrame({
    'A': [1, 2, np.nan, 4, 5],
    'B': [10, np.nan, 30, 40, 50],
    'C': ['X', 'Y', np.nan, 'Y', 'Z']
})

print("Original data with NaN:")
print(test_data)

# Method 1: Fill with constant value
print("\nFill with constant values:")
print(test_data.fillna(0))

Original data with NaN:
     A     B    C
0  1.0  10.0    X
1  2.0   NaN    Y
2  NaN  30.0  NaN
3  4.0  40.0    Y
4  5.0  50.0    Z

Fill with constant values:
     A     B  C
0  1.0  10.0  X
1  2.0   0.0  Y
2  0.0  30.0  0
3  4.0  40.0  Y
4  5.0  50.0  Z


In [73]:
# Method 2: Forward fill (carry forward)
print("Forward fill (ffill):")
print(test_data.fillna(method='ffill'))

# Method 3: Backward fill
print("\nBackward fill (bfill):")
print(test_data.fillna(method='bfill'))

Forward fill (ffill):
     A     B  C
0  1.0  10.0  X
1  2.0  10.0  Y
2  2.0  30.0  Y
3  4.0  40.0  Y
4  5.0  50.0  Z

Backward fill (bfill):
     A     B  C
0  1.0  10.0  X
1  2.0  30.0  Y
2  4.0  30.0  Y
3  4.0  40.0  Y
4  5.0  50.0  Z


In [74]:
# Method 4: Fill with mean, median, or mode
numeric_only = test_data[['A', 'B']]
print("Fill with mean:")
print(numeric_only.fillna(numeric_only.mean()))

# Fill different columns with different values
print("\nFill columns differently:")
print(test_data.fillna({'A': 0, 'B': 999, 'C': 'Unknown'}))

Fill with mean:
     A     B
0  1.0  10.0
1  2.0  32.5
2  3.0  30.0
3  4.0  40.0
4  5.0  50.0

Fill columns differently:
     A      B        C
0  1.0   10.0        X
1  2.0  999.0        Y
2  0.0   30.0  Unknown
3  4.0   40.0        Y
4  5.0   50.0        Z


### 4C: Interpolation for Time Series

In [75]:
# Time series example
time_series = pd.Series([1, np.nan, np.nan, 4, 5, np.nan, 7])

print("Original:")
print(time_series.values)

# Linear interpolation
print("\nLinear interpolation:")
print(time_series.interpolate().values)

# Forward/Backward fill for interpolate
print("\nLinear then forward fill:")
print(time_series.interpolate(method='linear').fillna(method='ffill').values)

Original:
[ 1. nan nan  4.  5. nan  7.]

Linear interpolation:
[1. 2. 3. 4. 5. 6. 7.]

Linear then forward fill:
[1. 2. 3. 4. 5. 6. 7.]


---

## Part 5: Applying Functions

### 5A: apply() on Series

In [76]:
# Built-in functions
years = netflix['release_year'].dropna()

# Apply NumPy function
sqrt_years = years.apply(np.sqrt)
print(f"Square root of first 5 years: {sqrt_years.head().values}")

# Custom function
def decade(year):
    return f"{int(year//10)*10}s"

decades = years.apply(decade)
print(f"\nDecades: {decades.head().values}")

Square root of first 5 years: [44.98888752 44.91102315 44.97777229 44.92215489 44.91102315]

Decades: ['2020s' '2010s' '2020s' '2010s' '2010s']


In [77]:
# Lambda functions for quick transformations
titles = netflix['title'].head()

# Length of each title
title_lengths = titles.apply(lambda x: len(x) if pd.notna(x) else 0)
print("Title lengths:")
print(title_lengths.values)

Title lengths:
[12 10 15  7  9]


### 5B: apply() on DataFrames (Row-wise)

In [78]:
# Apply function to each row
sales_sample = pd.DataFrame({
    'Q1': [100, 200, 150],
    'Q2': [120, 210, 160],
    'Q3': [150, 250, 180],
    'Q4': [180, 280, 200]
}, index=['Product A', 'Product B', 'Product C'])

print("Quarterly sales:")
print(sales_sample)

# Apply sum across rows
annual_sales = sales_sample.apply(lambda row: row.sum(), axis=1)
print("\nAnnual sales:")
print(annual_sales)

# Apply mean across rows
avg_sales = sales_sample.apply(lambda row: row.mean(), axis=1)
print("\nAverage quarterly sales:")
print(avg_sales)

Quarterly sales:
            Q1   Q2   Q3   Q4
Product A  100  120  150  180
Product B  200  210  250  280
Product C  150  160  180  200

Annual sales:
Product A    550
Product B    940
Product C    690
dtype: int64

Average quarterly sales:
Product A    137.5
Product B    235.0
Product C    172.5
dtype: float64


> ⚠️ **Common Pitfall**: When applying row-wise, use `axis=1`. Default `axis=0` applies to columns. Test with `.apply(print)` if confused!

---

## Part 6: Extreme Values

In [79]:
# Find top N values
print("Top 5 most recent Netflix content:")
top_years = netflix.nlargest(5, 'release_year')[['title', 'type', 'release_year']]
print(top_years)

# Find bottom N values
print("\n5 oldest Netflix content:")
oldest = netflix.nsmallest(5, 'release_year')[['title', 'type', 'release_year']]
print(oldest)

Top 5 most recent Netflix content:
              title     type  release_year
0      Breaking Bad    Movie          2024
19        The Crown  TV Show          2024
32  Stranger Things  TV Show          2024
34        The Crown  TV Show          2024
47  Stranger Things    Movie          2024

5 oldest Netflix content:
              title     type  release_year
11            Ozark    Movie          2015
13  Handmaid's Tale    Movie          2015
42   Peaky Blinders    Movie          2015
56            Ozark  TV Show          2015
17  Stranger Things    Movie          2016


In [80]:
# Multiple columns
if 'Sales' in superstore.columns and 'Profit' in superstore.columns:
    print("Top 10 products by profit:")
    top_profit = superstore.nlargest(10, 'Profit')
    print(top_profit[['Product Name', 'Sales', 'Profit']].drop_duplicates().head())

Top 10 products by profit:
   Product Name       Sales      Profit
54       Prod_D  302.970990  147.130091
39       Prod_B  225.674722  144.356417
40       Prod_D   69.798735  142.489459
34       Prod_A  483.159696  138.581941
78       Prod_B  185.648207  137.345998


---

## Part 7: Statistical Analysis

### 7A: Correlation Analysis

In [81]:
# Correlation between numerical columns
if 'Sales' in superstore.columns and 'Profit' in superstore.columns:
    correlation = superstore[['Sales', 'Profit', 'Quantity']].corr()
    print("Correlation Matrix:")
    print(correlation)
    
    # Specific correlation
    sales_profit_corr = superstore['Sales'].corr(superstore['Profit'])
    print(f"\nSales-Profit correlation: {sales_profit_corr:.3f}")

Correlation Matrix:
             Sales    Profit  Quantity
Sales     1.000000 -0.034033  0.239258
Profit   -0.034033  1.000000 -0.015123
Quantity  0.239258 -0.015123  1.000000

Sales-Profit correlation: -0.034


### 7B: Descriptive Statistics

In [82]:
# Detailed stats
if 'Sales' in superstore.columns:
    print("Superstore Sales Statistics:")
    print(superstore['Sales'].describe())

Superstore Sales Statistics:
count    100.000000
mean     240.388564
std      145.769811
min       12.705837
25%      104.668373
50%      237.429803
75%      367.799528
max      493.574599
Name: Sales, dtype: float64


In [83]:
# Percentiles
if 'Sales' in superstore.columns:
    sales = superstore['Sales']
    print(f"25th percentile: {sales.quantile(0.25):.0f}")
    print(f"50th percentile (median): {sales.quantile(0.50):.0f}")
    print(f"75th percentile: {sales.quantile(0.75):.0f}")
    print(f"95th percentile: {sales.quantile(0.95):.0f}")

25th percentile: 105
50th percentile (median): 237
75th percentile: 368
95th percentile: 475


---

## Part 8: Real-World Analysis - Google Play Store

In [84]:
# Explore Google Play Store data
print("Google Play Store columns:")
print(googleplay.columns.tolist())
print(f"\nDataset shape: {googleplay.shape}")
print(f"\nFirst few rows:")
print(googleplay.head())

Google Play Store columns:
['App', 'Category', 'Rating', 'Reviews', 'Installs']

Dataset shape: (80, 5)

First few rows:
     App Category    Rating  Reviews  Installs
0  App_0     News  3.566730    25451     90780
1  App_1    Music  4.309981    88768    153617
2  App_2    Games  2.647463    15405    546977
3  App_3     News  3.868671    75453    798606
4  App_4     News  2.256042    76897    351605


In [85]:
# Top app categories
print("Top 10 app categories:")
if 'Category' in googleplay.columns:
    categories = googleplay['Category'].value_counts().head(10)
    print(categories)

Top 10 app categories:
Category
News            26
Productivity    17
Music           15
Games           11
Social          11
Name: count, dtype: int64


In [86]:
# Average rating by category
if 'Category' in googleplay.columns and 'Rating' in googleplay.columns:
    print("Average rating by top 10 categories:")
    ratings_by_cat = googleplay.groupby('Category')['Rating'].mean().nlargest(10)
    print(ratings_by_cat.round(2))

Average rating by top 10 categories:
Category
Social          3.47
Productivity    3.44
Music           3.41
News            3.38
Games           3.09
Name: Rating, dtype: float64


---

## Part 9: Try It Yourself!

### Challenge 1: Netflix Data Analysis

1. Find the top 10 most common ratings
2. Count missing values in each column
3. Remove duplicates and report how many were removed
4. Sort by title (alphabetically) and show first 5

In [87]:
# Challenge 1: Netflix Data Analysis

print("NETFLIX METHODS ANALYSIS")
print("=" * 60)

# Task 1: Top 10 most common ratings
print("\n1️⃣ Top 10 Most Common Ratings:")
top_ratings = netflix['rating'].value_counts().head(10)
print(top_ratings)

# Task 2: Count missing values
print("\n2️⃣ Missing Values by Column:")
missing_values = netflix.isnull().sum()
missing_pct = (netflix.isnull().sum() / len(netflix) * 100).round(2)
print(f"\nAll columns:")
print(missing_values)
print(f"\nPercentage missing:")
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

# Task 3: Remove duplicates
print("\n3️⃣ Duplicate Analysis:")
before = len(netflix)
netflix_dedup = netflix.drop_duplicates()
after = len(netflix_dedup)
print(f"Rows before: {before}")
print(f"Rows after: {after}")
print(f"Duplicates removed: {before - after}")

# Task 4: Sort and show first 5
print("\n4️⃣ Alphabetically sorted titles (first 10):")
netflix_sorted = netflix.sort_values('title')
print(netflix_sorted[['title', 'type', 'release_year']].head(10))

print("\n📊 Summary:")
print(f"• Most common rating: {top_ratings.index[0]} ({top_ratings.values[0]} titles)")
print(f"• Missing data: {missing_values[missing_values > 0].sum()} total missing values")
print(f"• Data is {(100 * after / before):.1f}% unique")


NETFLIX METHODS ANALYSIS

1️⃣ Top 10 Most Common Ratings:
rating
PG       14
TV-14    14
TV-MA    12
PG-13    11
R         9
Name: count, dtype: int64

2️⃣ Missing Values by Column:

All columns:
title           0
type            0
release_year    0
rating          0
dtype: int64

Percentage missing:
Series([], dtype: float64)

3️⃣ Duplicate Analysis:
Rows before: 60
Rows after: 60
Duplicates removed: 0

4️⃣ Alphabetically sorted titles (first 10):
           title     type  release_year
29     Bodyguard    Movie          2023
44     Bodyguard    Movie          2017
14     Bodyguard    Movie          2020
59     Bodyguard    Movie          2019
0   Breaking Bad    Movie          2024
30  Breaking Bad  TV Show          2023
15  Breaking Bad    Movie          2021
45  Breaking Bad    Movie          2017
22     Chernobyl  TV Show          2022
52     Chernobyl  TV Show          2022

📊 Summary:
• Most common rating: PG (14 titles)
• Missing data: 0 total missing values
• Data is 100.0% un

### Challenge 2: Superstore Sales Analysis

1. Find the top 5 products by profit
2. Calculate average profit per region
3. Sort regions by average profit
4. Find correlation between Sales and Profit

In [88]:
# Challenge 2: Superstore Sales Analysis

print("\nSUPERSTORE SALES ANALYSIS")
print("=" * 60)

# Task 1: Top 5 products by profit
print("\n1️⃣ Top 5 Most Profitable Products:")
top_products = superstore.nlargest(5, 'Profit')[['Product Name', 'Sales', 'Profit']]
print(top_products)

# Task 2: Average profit per region
print("\n2️⃣ Average Profit by Region:")
avg_profit = superstore.groupby('Region')['Profit'].mean().round(2)
print(avg_profit)

# Task 3: Sort regions by profit
print("\n3️⃣ Regions Ranked by Average Profit:")
sorted_regions = superstore.groupby('Region')['Profit'].mean().sort_values(ascending=False)
for i, (region, profit) in enumerate(sorted_regions.items(), 1):
    print(f"   {i}. {region}: ${profit:.2f}")

# Task 4: Correlation between Sales and Profit
print("\n4️⃣ Sales vs Profit Correlation:")
correlation = superstore['Sales'].corr(superstore['Profit'])
print(f"Correlation coefficient: {correlation:.4f}")
print("\nInterpretation:")
if correlation > 0.8:
    print("✓ STRONG positive correlation: Higher sales almost always mean higher profit")
elif correlation > 0.5:
    print("✓ MODERATE positive correlation: Higher sales generally lead to higher profit")
elif correlation > 0:
    print("✓ WEAK positive correlation: Some relationship between sales and profit")
else:
    print("✗ Negative correlation: Concerning - higher sales associates with lower profit!")

print("\n💰 Additional Insights:")
print(f"• Most profitable region: {sorted_regions.index[0]} (${sorted_regions.values[0]:.2f} avg)")
print(f"• Average profit per sale: ${superstore['Profit'].mean():.2f}")
print(f"• Profit margin (avg): {(superstore['Profit'].sum() / superstore['Sales'].sum() * 100):.1f}%")



SUPERSTORE SALES ANALYSIS

1️⃣ Top 5 Most Profitable Products:
   Product Name       Sales      Profit
54       Prod_D  302.970990  147.130091
39       Prod_B  225.674722  144.356417
40       Prod_D   69.798735  142.489459
34       Prod_A  483.159696  138.581941
78       Prod_B  185.648207  137.345998

2️⃣ Average Profit by Region:
Region
East     59.85
North    50.00
South    44.31
West     38.43
Name: Profit, dtype: float64

3️⃣ Regions Ranked by Average Profit:
   1. East: $59.85
   2. North: $50.00
   3. South: $44.31
   4. West: $38.43

4️⃣ Sales vs Profit Correlation:
Correlation coefficient: -0.0340

Interpretation:
✗ Negative correlation: Concerning - higher sales associates with lower profit!

💰 Additional Insights:
• Most profitable region: East ($59.85 avg)
• Average profit per sale: $49.57
• Profit margin (avg): 20.6%


---

## Part 10: Summary & Key Takeaways

### ✅ Checklist - DataFrame Methods Mastery

- [ ] I can sort data with `.sort_values()` in multiple ways
- [ ] I understand `.rank()` and when to use it
- [ ] I can use `.value_counts()` for frequency analysis
- [ ] I find and remove duplicates with `.drop_duplicates()`
- [ ] I handle missing data with `.fillna()` and `.interpolate()`
- [ ] I apply custom functions with `.apply()`
- [ ] I understand `.nlargest()` and `.nsmallest()`
- [ ] I calculate correlation with `.corr()`

### Essential Methods Quick Reference

| Method | Purpose | Example |
|--------|---------|----------|
| `.sort_values()` | Sort DataFrame | `df.sort_values('col')` |
| `.rank()` | Rank values | `df['col'].rank()` |
| `.value_counts()` | Frequency | `df['col'].value_counts()` |
| `.unique()` | Unique values | `df['col'].unique()` |
| `.drop_duplicates()` | Remove duplicates | `df.drop_duplicates()` |
| `.fillna()` | Fill missing | `df.fillna(0)` |
| `.interpolate()` | Interpolate | `df['col'].interpolate()` |
| `.apply()` | Apply function | `df['col'].apply(func)` |
| `.nlargest()` | Top N | `df.nlargest(5, 'col')` |
| `.corr()` | Correlation | `df.corr()` |

---

## 🚀 Next Steps

**Excellent! You now handle data analysis and transformation!** Ready for advanced techniques:

**→ Advanced Topics (GroupBy, Merge, MultiIndex)**
- Group data and aggregate
- Split-apply-combine workflow
- Merge/join datasets
- Multi-level indexing

---

**You've covered essential data manipulation techniques!** 🎉